# Question 3 (20 points)

<h2 style="font-weight: bold;">There are many ways to measure a node's centrality in a network. Pick one centrality measure out of {<s>eigenvector centrality</s>, Katz centrality, <s>closeness centrality</s>, betweenness centrality, PageRank, etc.} and do the following:</h2>

<h2 style="font-weight: bold;">a) Write your own function to compute a single node's centrality in a graph. This function should take a <code>networkx</code> graph and one node in the graph as input. As output, it should return a float or integer representing the node's centrality in the graph according to one of the centrality measures listed above.</h2>

# Percolation Centrality 
Based on the paper by [Piraveenan et al. (2013)](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0053095), this measure quantifies node importance during network percolation processes.
## The Problem

During contagion spread in networks (epidemics, computer viruses, information diffusion), nodes have different states: some are infected, others aren't. Traditional centrality measures like betweenness only consider network topology, ignoring which nodes are already infected. This is a critical limitation when allocating limited resources during an outbreak.

## The Mathematical Definition

Percolation centrality of node $v$ at time $t$:

$$PC^t(v) = \frac{1}{N-2} \sum_{s \neq v \neq r} \frac{\sigma_{s,r}(v)}{\sigma_{s,r}} \cdot \frac{x_s^t}{\sum_{i=1}^{N} x_i^t - x_v^t}$$

**Components breakdown:**

1. **$\frac{\sigma_{s,r}(v)}{\sigma_{s,r}}$**: Fraction of shortest paths from source $s$ to target $r$ passing through $v$ (this is standard betweenness)

2. **$x_s^t \in [0,1]$**: Percolation state of source node $s$ at time $t$
   - 0 = not infected
   - 1 = fully infected  
   - 0 < x < 1 = partially infected

3. **$\frac{x_s^t}{\sum_{i=1}^{N} x_i^t - x_v^t}$**: Weight based on source percolation state, normalized across all nodes except $v$

## Why This Matters

The weight term means paths from infected nodes contribute more to the centrality score. This captures a key insight: infected nodes are the ones actually spreading the contagion, so paths originating from them are more important.

In [28]:
import networkx as nx
import numpy as np
from pyvis.network import Network
import matplotlib.cm as cm
import pandas as pd

In [9]:
def percolation_centrality(G):
    """
    Compute Percolation Centrality (PC) for each node in the graph G,
    using node degree as the percolation state (x_i).
    
    Parameters
    ----------
    G : networkx.Graph
        Input graph.
    
    Returns
    -------
    pc_dict : dict
        Dictionary mapping each node to its percolation centrality value.
    """
    
    N = len(G.nodes)
    
    # Percolation state: use degree of each node
    x = {node: G.degree(node) for node in G.nodes}
    
    # Precompute denominator for weights
    total_percolation = sum(x.values())
    
    # Initialize percolation centrality dictionary
    pc_dict = {v: 0.0 for v in G.nodes}
    
    # Loop over all pairs (s, r) with s != r
    for s in G.nodes:
        for r in G.nodes:
            if s == r:
                continue
            
            # Get all shortest paths between s and r
            paths = list(nx.all_shortest_paths(G, source=s, target=r))
            sigma_sr = len(paths)  # number of shortest paths
            
            if sigma_sr == 0:
                continue
            
            # Weight contribution depends on percolation state of source s
            for v in G.nodes:
                if v in (s, r):
                    continue
                
                # Count how many shortest paths go through v
                sigma_sr_v = sum(1 for path in paths if v in path[1:-1])
                
                # Avoid division by zero
                denom = total_percolation - x[v]
                if denom <= 0:
                    continue
                
                w_sv = x[s] / denom
                
                # Update PC
                pc_dict[v] += (sigma_sr_v / sigma_sr) * w_sv
    
    # Normalization factor 1 / (N - 2)
    for v in pc_dict:
        pc_dict[v] /= (N - 2)
    
    return pc_dict

In [32]:
# Example: Karate Club graph
G = nx.karate_club_graph()

# Custom percolation centrality (assuming you already defined percolation_centrality)
pc_custom = percolation_centrality(G)

# Build DataFrame
df_pc = pd.DataFrame({
    "Node": list(pc_custom.keys()),
    "Percolation Centrality": list(pc_custom.values())
})

df_pc

,Node,Percolation Centrality
0,0,0.378328
1,1,0.045450
2,2,0.138347
3,3,0.009676
4,4,0.000477
5,5,0.021861
6,6,0.021861
7,7,0.000000
8,8,0.056573
9,9,0.001647


In [18]:
# Normalize PC values for mapping into colormap and sizes
pc_values = list(pc.values())
min_pc, max_pc = min(pc_values), max(pc_values)

def normalize(value, vmin, vmax):
    """Return normalized value in [0,1]. If all values equal, return 0.5."""
    if vmax == vmin:
        return 0.5
    return (value - vmin) / (vmax - vmin)

# Choose a perceptually-uniform colormap
colormap_name = "viridis"     # alternatives: "plasma", "cividis", "magma"
cmap = cm.get_cmap(colormap_name)

# Create Pyvis network (set notebook=True if you're in Jupyter)
net = Network(notebook=True, height="700px", width="100%", bgcolor="white", font_color="black")
net.from_nx(G)

# Style nodes: color from colormap, size proportional to PC
for node in net.nodes:
    n = node["id"]
    score = pc[n]
    norm_score = normalize(score, min_pc, max_pc)
    
    # cmap returns (r,g,b,a) floats in [0,1] -> convert to 0-255 integers
    r, g, b, a = cmap(norm_score)
    r_i, g_i, b_i = int(255*r), int(255*g), int(255*b)
    color = f"rgb({r_i},{g_i},{b_i})"
    
    # Node size (tweak base and scale as you like)
    size = 12 + norm_score * 36
    
    node["color"] = color
    node["size"] = size
    node["title"] = f"Node {n}<br>PC: {score:.4f}"

# Save & show interactive HTML
net.show("karate_pc_viridis.html")

karate_pc_viridis.html


<ipython-input-18-bb947c8ebf1c>:13: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = cm.get_cmap(colormap_name)


<h2 style="font-weight: bold;">(b) Augment your function with error handling. It should be able to return useful messages when given any of the following:</h2>

<h3 style="font-weight: bold; margin-left: 20px;">i. A node that is not in the graph</h3>

<h3 style="font-weight: bold; margin-left: 20px;">ii. A graph that is not a <code>networkx</code> object</h3>

<h3 style="font-weight: bold; margin-left: 20px;">iii. A graph with no connectivity (or not enough connectivity to produce a valid centrality result)</h3>

<h3 style="font-weight: bold; margin-left: 20px;">iv. Any situation that might produce numerical difficulties (e.g., dividing by zero) for your chosen centrality measure</h3>

In [20]:
def percolation_centrality(G, nodes=None):
    """
    Compute Percolation Centrality (PC) for each node in the graph G,
    using node degree as the percolation state (x_i).

    Error handling is included for:
        i.   Invalid nodes
        ii.  Input not being a NetworkX graph
        iii. Graphs with insufficient connectivity
        iv.  Numerical difficulties (e.g., division by zero)

    Parameters
    ----------
    G : networkx.Graph
        Input graph.
    nodes : list or None
        Subset of nodes to compute PC for. If None, compute for all nodes.

    Returns
    -------
    pc_dict : dict
        Dictionary mapping each node to its percolation centrality value.
    """

    # --- (ii) Check that input is a NetworkX graph ---
    if not isinstance(G, nx.Graph):
        raise TypeError("Input must be a networkx.Graph object (or subclass).")

    N = len(G.nodes)
    if N < 3:
        # At least 3 nodes are needed (since normalization uses N-2)
        raise ValueError("Graph must have at least 3 nodes to compute percolation centrality.")

    # --- (iii) Check connectivity ---
    if not nx.is_connected(G):
        raise ValueError("Graph is not connected. Percolation centrality is undefined.")

    # --- (i) Check if requested nodes exist ---
    if nodes is not None:
        invalid_nodes = [n for n in nodes if n not in G]
        if invalid_nodes:
            raise ValueError(f"The following nodes are not in the graph: {invalid_nodes}")
        node_list = nodes
    else:
        node_list = list(G.nodes)

    # Percolation state: use node degree
    x = {node: G.degree(node) for node in G.nodes}
    total_percolation = sum(x.values())

    # Guard against numerical problems
    if total_percolation <= 0:
        raise ValueError("Total percolation state is non-positive. Cannot compute centrality.")

    # Initialize PC values
    pc_dict = {v: 0.0 for v in node_list}

    # Loop over all pairs (s, r) with s != r
    for s in G.nodes:
        for r in G.nodes:
            if s == r:
                continue

            # Get all shortest paths between s and r
            try:
                paths = list(nx.all_shortest_paths(G, source=s, target=r))
            except nx.NetworkXNoPath:
                # Should not happen if the graph is connected, but check just in case
                continue

            sigma_sr = len(paths)
            if sigma_sr == 0:
                continue

            for v in node_list:
                if v in (s, r):
                    continue

                # Count how many shortest paths go through v
                sigma_sr_v = sum(1 for path in paths if v in path[1:-1])

                # --- (iv) Avoid division by zero ---
                denom = total_percolation - x[v]
                if denom <= 0:
                    continue  # skip contribution safely

                w_sv = x[s] / denom
                pc_dict[v] += (sigma_sr_v / sigma_sr) * w_sv

    # Normalize
    for v in pc_dict:
        pc_dict[v] /= (N - 2)

    return pc_dict

In [21]:
# --- i. A node that is not in the graph ---
try:
    G = nx.karate_club_graph()
    percolation_centrality(G, nodes=[100])  # Node 100 does not exist
except Exception as e:
    print("Error (i):", e)


# --- ii. A graph that is not a networkx object ---
try:
    percolation_centrality("not_a_graph")  # Invalid type
except Exception as e:
    print("Error (ii):", e)


# --- iii. A graph with no connectivity ---
try:
    G = nx.Graph()
    G.add_nodes_from([1, 2, 3, 4])  # No edges, disconnected
    percolation_centrality(G)
except Exception as e:
    print("Error (iii):", e)


# --- iii bis. A graph with too few nodes ---
try:
    G = nx.path_graph(2)  # Only 2 nodes (N-2 = 0 → invalid)
    percolation_centrality(G)
except Exception as e:
    print("Error (iii bis):", e)


# --- iv. Numerical difficulties (division by zero) ---
try:
    G = nx.Graph()
    # Build a "star" graph with isolated degree zero node to force denominator = 0
    G.add_nodes_from([0, 1, 2])
    G.add_edges_from([(0, 1)])  # Node 2 is isolated (degree 0)
    percolation_centrality(G)
except Exception as e:
    print("Error (iv):", e)

Error (i): The following nodes are not in the graph: [100]
Error (ii): Input must be a networkx.Graph object (or subclass).
Error (iii): Graph is not connected. Percolation centrality is undefined.
Error (iii bis): Graph must have at least 3 nodes to compute percolation centrality.
Error (iv): Graph is not connected. Percolation centrality is undefined.


<h2 style="font-weight: bold;"> c) Compare your function to the corresponding <code>networkx</code> function across several graphs you design as test inputs, including one with no connectivity. Does your function ever return different results from the <code>networkx</code> function? Why do you think this is the case?</h2>

In [26]:
def compare_pc(G, name):
    print(f"\n--- {name} ---")
    try:
        pc_custom = percolation_centrality(G)
    except Exception as e:
        print("Custom function error:", e)
        pc_custom = None
    
    try:
        states = {n: G.degree(n) for n in G.nodes}
        pc_nx = nx.percolation_centrality(G, states=states, weight=None)
    except Exception as e:
        print("NetworkX function error:", e)
        pc_nx = None
    
    if pc_custom is not None and pc_nx is not None:
        # Crea un DataFrame
        df = pd.DataFrame({
            'Node': list(G.nodes),
            'Custom_PC': [pc_custom[n] for n in G.nodes],
            'NetworkX_PC': [pc_nx[n] for n in G.nodes]
        })
        # Mostra i primi 10 nodi per grafi grandi
        display(df.head(10))
        return df

In [31]:
# --- Test Graphs ---

# 1. Karate Club Graph (small social network)
G1 = nx.karate_club_graph()
compare_pc(G1, "Karate Club Graph")

# 2. Path Graph (chain structure)
G2 = nx.path_graph(10)
compare_pc(G2, "Path Graph (10 nodes)")

# 3. Star Graph
G3 = nx.star_graph(8)
compare_pc(G3, "Star Graph (center + 8 leaves)")

# 4. Erdős–Rényi random graph
G4 = nx.erdos_renyi_graph(30, 0.1, seed=42)
if nx.is_connected(G4):
    compare_pc(G4, "Erdős–Rényi Random Graph (n=30, p=0.1)")
else:
    print("\n--- Erdős–Rényi Random Graph ---")
    try:
        percolation_centrality(G4)
    except Exception as e:
        print("Custom:", e)

# 5. Barabási–Albert scale-free network
G5 = nx.barabasi_albert_graph(30, 2, seed=42)
compare_pc(G5, "Barabási–Albert Scale-Free Graph (n=30, m=2)")

# 6. Watts–Strogatz small-world network
G6 = nx.watts_strogatz_graph(30, k=4, p=0.2, seed=42)
compare_pc(G6, "Watts–Strogatz Small-World Graph (n=30, k=4, p=0.2)")

# 7. Disconnected graph (no edges)
G7 = nx.empty_graph(5)
compare_pc(G7, "Disconnected Graph (no edges)")


--- Karate Club Graph ---


,Node,Custom_PC,NetworkX_PC
0,0,0.378328,0.378328
1,1,0.045450,0.045450
2,2,0.138347,0.138347
3,3,0.009676,0.009676
4,4,0.000477,0.000477
5,5,0.021861,0.021861
6,6,0.021861,0.021861
7,7,0.000000,0.000000
8,8,0.056573,0.056573
9,9,0.001647,0.001647



--- Path Graph (10 nodes) ---


,Node,Custom_PC,NetworkX_PC
0,0,0.000000,0.000000
1,1,0.179688,0.179688
2,2,0.367188,0.367188
3,3,0.492188,0.492188
4,4,0.554688,0.554688
5,5,0.554688,0.554688
6,6,0.492188,0.492188
7,7,0.367188,0.367188
8,8,0.179688,0.179688
9,9,0.000000,0.000000



--- Star Graph (center + 8 leaves) ---


...


--- Erdős–Rényi Random Graph ---
Custom: Graph is not connected. Percolation centrality is undefined.

--- Barabási–Albert Scale-Free Graph (n=30, m=2) ---


,Node,Custom_PC,NetworkX_PC
0,0,0.398165,0.398165
1,1,0.223827,0.223827
2,2,0.019975,0.019975
3,3,0.006007,0.006007
4,4,0.160443,0.160443
5,5,0.063352,0.063352
6,6,0.050926,0.050926
7,7,0.000000,0.000000
8,8,0.044668,0.044668
9,9,0.010606,0.010606



--- Watts–Strogatz Small-World Graph (n=30, k=4, p=0.2) ---


,Node,Custom_PC,NetworkX_PC
0,0,0.012388,0.012388
1,1,0.051465,0.051465
2,2,0.041651,0.041651
3,3,0.115210,0.115210
4,4,0.015583,0.015583
5,5,0.032497,0.032497
6,6,0.219209,0.219209
7,7,0.067246,0.067246
8,8,0.021612,0.021612
9,9,0.042103,0.042103



--- Disconnected Graph (no edges) ---
Custom function error: Graph is not connected. Percolation centrality is undefined.


<h2 style="font-weight: normal;">(Surprisingly) The values predicted by my function and those predicted by NetworkX are identical for each node in every network used for testing.<\h2>

<h2 style="font-weight: bold;">Putting it all together: Write a function that takes a centrality function as input and tests the function for robustness to edge cases (like a graph with no connectivity) and correctness of outputs. Explain why you selected the test cases you chose.<\h2>

I selected these networks because they cover different topological patterns and stress-test edge cases:

- **Karate Club Graph** – classic benchmark, widely used to validate centrality measures.
- **Path Graph (10 nodes)** – linear structure, tests whether middle nodes are recognized as more central.
- **Star Graph (center + leaves)** – strong hub-and-spoke pattern; center should dominate centrality.
- **Cycle Graph (10 nodes)** – symmetric structure where all nodes should have equal centrality.
- **Complete Graph (10 nodes)** – fully connected, all nodes identical; ensures normalization is correct.
- **Balanced Tree (r=2, h=3)** – hierarchical structure, useful for testing centrality in branching networks.
- **Erdős–Rényi Graph (n=30, p=0.2)** – random network, good to test robustness under stochastic topology.
- **Barabási–Albert Graph (n=30, m=2)** – scale-free structure with hubs, tests behavior on heterogeneous degree distributions.
- **Watts–Strogatz Graph (n=30, k=4, p=0.2)** – small-world network, balances clustering and short paths.
- **Ladder Graph (10 rungs)** – structured, nearly-planar graph, intermediate between path and grid.
- **Disconnected Graph (5 isolated nodes)** – edge case where percolation centrality should be undefined (robustness test).


In [39]:
def test_percolation_centrality_styled(custom_func, tol=1e-8):
    """
    Test custom percolation centrality against NetworkX implementation
    and display DataFrames with rows highlighted where the results are effectively equal.
    """

    # Define graphs
    test_graphs = {
        "Karate Club Graph": nx.karate_club_graph(),
        "Path Graph (10 nodes)": nx.path_graph(10),
        "Star Graph (center + 8 leaves)": nx.star_graph(8),
        "Cycle Graph (10 nodes)": nx.cycle_graph(10),
        "Complete Graph (10 nodes)": nx.complete_graph(10),
        "Balanced Tree (r=2, h=3)": nx.balanced_tree(2, 3),
        "Erdős–Rényi Graph (n=30, p=0.2)": nx.erdos_renyi_graph(30, 0.2, seed=42),
        "Barabási–Albert Graph (n=30, m=2)": nx.barabasi_albert_graph(30, 2, seed=42),
        "Watts–Strogatz Graph (n=30, k=4, p=0.2)": nx.watts_strogatz_graph(30, 4, 0.2, seed=42),
        "Ladder Graph (10 rungs)": nx.ladder_graph(10),
        "Disconnected Graph (5 nodes, no edges)": nx.empty_graph(5)
    }

    results = {}

    for name, G in test_graphs.items():
        print(f"\n--- {name} ---")
        try:
            # Compute centralities
            pc_custom = custom_func(G)
            states = {n: G.degree(n) for n in G.nodes}
            pc_nx = nx.percolation_centrality(G, states=states, weight=None)

            # Create DataFrame
            df = pd.DataFrame({
                "Node": list(G.nodes),
                "Custom_PC": [pc_custom[n] for n in G.nodes],
                "NetworkX_PC": [pc_nx[n] for n in G.nodes]
            }).sort_values(by="Custom_PC", ascending=False).reset_index(drop=True)

            # Highlight rows where results are effectively equal
            def highlight_equal(row):
                return ['background-color: lightgreen' if abs(row['Custom_PC'] - row['NetworkX_PC']) < tol else '' for _ in row]

            styled_df = df.style.apply(highlight_equal, axis=1)
            display(styled_df)

            results[name] = df

        except Exception as e:
            print("Error:", e)
            results[name] = f"Error: {e}"

    return results

# Usage
results = test_percolation_centrality_styled(percolation_centrality)


--- Karate Club Graph ---


,Node,Custom_PC,NetworkX_PC
0,0,0.378328,0.378328
1,33,0.242389,0.242389
2,2,0.138347,0.138347
3,32,0.124141,0.124141
4,31,0.123032,0.123032
5,8,0.056573,0.056573
6,13,0.049909,0.049909
7,1,0.045450,0.045450
8,19,0.033375,0.033375
9,27,0.022725,0.022725



--- Path Graph (10 nodes) ---


,Node,Custom_PC,NetworkX_PC
0,4,0.554688,0.554688
1,5,0.554688,0.554688
2,3,0.492188,0.492188
3,6,0.492188,0.492188
4,2,0.367188,0.367188
5,7,0.367188,0.367188
6,1,0.179688,0.179688
7,8,0.179688,0.179688
8,0,0.000000,0.000000
9,9,0.000000,0.000000



--- Star Graph (center + 8 leaves) ---


...


--- Watts–Strogatz Graph (n=30, k=4, p=0.2) ---


,Node,Custom_PC,NetworkX_PC
0,6,0.219209,0.219209
1,24,0.164517,0.164517
2,13,0.130888,0.130888
3,3,0.115210,0.115210
4,26,0.105811,0.105811
5,14,0.104381,0.104381
6,28,0.089596,0.089596
7,20,0.072950,0.072950
8,10,0.071308,0.071308
9,18,0.067990,0.067990



--- Ladder Graph (10 rungs) ---


,Node,Custom_PC,NetworkX_PC
0,15,0.269518,0.269518
1,4,0.269518,0.269518
2,5,0.269518,0.269518
3,14,0.269518,0.269518
4,16,0.241968,0.241968
5,13,0.241968,0.241968
6,6,0.241968,0.241968
7,3,0.241968,0.241968
8,12,0.186677,0.186677
9,2,0.186677,0.186677



--- Disconnected Graph (5 nodes, no edges) ---
Error: Graph is not connected. Percolation centrality is undefined.
